In [ ]:
import pandas as pd

results = pd.read_csv('results.csv', index_col=0, sep=',')

In [ ]:
results

In [ ]:

import matplotlib.pyplot as plt


matrix = results.pivot(index="Experiment", columns="Run", values="E")

plt.imshow(matrix)
plt.colorbar(label="E (lower is better)")
plt.xticks(range(len(matrix.columns)), matrix.columns)
plt.yticks(range(len(matrix.index)), matrix.index)

plt.xlabel("Run")
plt.ylabel("Experiment")
plt.title("Matrix of E values (lower = better)")

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = results.copy()

# Run sicher als int
df["Run"] = pd.to_numeric(df["Run"], errors="coerce").astype("Int64")

run_map = {
    1: ("GPT 5.2", 0),
    2: ("GPT 5.2", 0.5),
    3: ("GPT 5.2", 1),
    4: ("CS 4.6", 0),
    5: ("CS 4.6", 0.5),
    6: ("CS 4.6", 1),
}

mapped = df["Run"].map(run_map)
df["Model"] = mapped.str[0]
df["Temp"] = mapped.str[1]

# Label sauber formatieren (damit 0.0 -> 0)
def fmt_temp(t):
    if pd.isna(t):
        return None
    return str(int(t)) if float(t).is_integer() else str(t)

df["Label"] = df["Model"] + "\n t = " + df["Temp"].map(fmt_temp)

order = [
    "GPT 5.2\n t = 0",
    "GPT 5.2\n t = 0.5",
    "GPT 5.2\n t = 1",
    "CS 4.6\n t = 0",
    "CS 4.6\n t = 0.5",
    "CS 4.6\n t = 1",
]
matrix = (
    df.pivot(index="Experiment", columns="Label", values="E")
      .reindex(columns=order)   # <- statt [order]
      .sort_index()
)

sns.set_theme(style="white", context="paper", font_scale=1.1)
plt.figure(figsize=(7, 4))

sns.heatmap(
    matrix,
    annot=True,
    fmt=".0f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    cbar_kws={"label": "Error sum E"},
)

plt.title("Error sum E for all experiments")
plt.xlabel("Model and Temperature")
plt.ylabel("Experiment")
plt.tight_layout()
plt.savefig("E.png", dpi=600, bbox_inches="tight")
plt.savefig("E.pdf", bbox_inches="tight")
plt.show()

In [ ]:
df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = results.copy()
df = df[df["Experiment"].isin([2, 3, 4, 5])]

df["Run"] = pd.to_numeric(df["Run"], errors="coerce").astype("Int64")

run_map = {
    1: ("GPT 5.2", 0),
    2: ("GPT 5.2", 0.5),
    3: ("GPT 5.2", 1),
    4: ("CS 4.6", 0),
    5: ("CS 4.6", 0.5),
    6: ("CS 4.6", 1),
}

mapped = df["Run"].map(run_map)
df["Model"] = mapped.str[0]
df["Temp"] = mapped.str[1]

def fmt_temp(t):
    if pd.isna(t):
        return None
    return str(int(t)) if float(t).is_integer() else str(t)

df["Label"] = df["Model"] + "\n t = " + df["Temp"].map(fmt_temp)

order = [
    "GPT 5.2\n t = 0",
    "GPT 5.2\n t = 0.5",
    "GPT 5.2\n t = 1",
    "CS 4.6\n t = 0",
    "CS 4.6\n t = 0.5",
    "CS 4.6\n t = 1",
]
matrix = (
    df.pivot(index="Experiment", columns="Label", values="Ec")
      .reindex(columns=order)   # <- statt [order]
      .sort_index()
)

sns.set_theme(style="white", context="paper", font_scale=1.1)
plt.figure(figsize=(7, 4))

sns.heatmap(
    matrix,
    annot=True,
    fmt=".0f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    cbar_kws={"label": "Confidence-weighted error sum Ec"},
)

plt.title("Confidence-weighted error for experiments 2-5")
plt.xlabel("Model and Temperature")
plt.ylabel("Experiment")
plt.tight_layout()
plt.savefig("Ec.png", dpi=600, bbox_inches="tight")
plt.savefig("Ec.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df = results.copy()


df["Run"] = pd.to_numeric(df["Run"], errors="coerce").astype("Int64")


run_map = {
    1: ("GPT 5.2", 0),
    2: ("GPT 5.2", 0.5),
    3: ("GPT 5.2", 1),
    4: ("CS 4.6", 0),
    5: ("CS 4.6", 0.5),
    6: ("CS 4.6", 1),
}

mapped = df["Run"].map(run_map)
df["Model"] = mapped.str[0]
df["Temp"] = mapped.str[1]

# --- Temp-Format (0.0 -> 0) ---
def fmt_temp(t):
    if pd.isna(t):
        return None
    return str(int(t)) if float(t).is_integer() else str(t)

# --- X-Achsenlabel: Model in Zeile 1, t in Zeile 2 ---
df["Label"] = df["Model"] + "\n t = " + df["Temp"].map(fmt_temp)

order = [
    "GPT 5.2\n t = 0",
    "GPT 5.2\n t = 0.5",
    "GPT 5.2\n t = 1",
    "CS 4.6\n t = 0",
    "CS 4.6\n t = 0.5",
    "CS 4.6\n t = 1",
]

# --- Matrix bauen: Experimente 1..5 beibehalten (1 ggf. als NaN placeholder) ---
matrix = (
    df.pivot(index="Experiment", columns="Label", values="Ec")
      .reindex(index=[1, 2, 3, 4, 5])     # Experiment 1 bleibt als Zeile drin
      .reindex(columns=order)             # feste Spaltenreihenfolge
)

# --- Mask für fehlende Werte (Experiment 1 wird dadurch grau) ---
mask = matrix.isna()

sns.set_theme(style="white", context="paper", font_scale=1.1)
plt.figure(figsize=(7, 4))

# 1) Heatmap für vorhandene Werte
sns.heatmap(
    matrix,
    annot=True,
    fmt=".0f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    mask=mask,
    cbar_kws={"label": "Confidence-weighted error sum Ec"},
)

# 2) Overlay: fehlende Werte grau darstellen
sns.heatmap(
    matrix,
    mask=~mask,
    cmap=["lightgrey"],
    cbar=False,
    linewidths=0.5,
)

plt.title("Confidence-weighted error for experiments 2–5")
plt.xlabel("Model and Temperature")
plt.ylabel("Experiment")

plt.tight_layout()
plt.savefig("Ec.png", dpi=600, bbox_inches="tight")
plt.savefig("Ec.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = results.copy()

# --- Runs sicher als int ---
df["Run"] = pd.to_numeric(df["Run"], errors="coerce").astype("Int64")

# --- Run -> (Model, Temp) Mapping ---
run_map = {
    1: ("GPT 5.2", 0),
    2: ("GPT 5.2", 0.5),
    3: ("GPT 5.2", 1),
    4: ("CS 4.6", 0),
    5: ("CS 4.6", 0.5),
    6: ("CS 4.6", 1),
}

mapped = df["Run"].map(run_map)
df["Model"] = mapped.str[0]
df["Temp"] = mapped.str[1]

def fmt_temp(t):
    if pd.isna(t):
        return None
    return str(int(t)) if float(t).is_integer() else str(t)

# --- X-Achsen Label ---
df["Label"] = df["Model"] + "\n t = " + df["Temp"].map(fmt_temp)

order = [
    "GPT 5.2\n t = 0",
    "GPT 5.2\n t = 0.5",
    "GPT 5.2\n t = 1",
    "CS 4.6\n t = 0",
    "CS 4.6\n t = 0.5",
    "CS 4.6\n t = 1",
]

# --- Matrix bauen (Exp 1 bleibt placeholder) ---
matrix = (
    df.pivot(index="Experiment", columns="Label", values="Ec")
      .reindex(index=[1,2,3,4,5])
      .reindex(columns=order)
)

# --- Colormap mit grauen NaN ---
cmap = plt.cm.get_cmap("RdYlGn_r").copy()
cmap.set_bad("lightgrey")

sns.set_theme(style="white", context="paper", font_scale=1.1)

plt.figure(figsize=(7,4))

sns.heatmap(
    matrix,
    annot=True,
    fmt=".0f",
    cmap=cmap,
    linewidths=0.5,
    cbar_kws={"label": "Confidence-weighted error sum Ec"},
)

plt.title("Confidence-weighted error for experiments 2–5")
plt.xlabel("Model and Temperature")
plt.ylabel("Experiment")

plt.tight_layout()

plt.savefig("Ec.png", dpi=600, bbox_inches="tight")
plt.savefig("Ec.pdf", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = results.copy()

# 1) Run sicher int
df["Run"] = pd.to_numeric(df["Run"], errors="coerce").astype("Int64")

# 2) Model/Temp Mapping
run_map = {
    1: ("GPT 5.2", 0),
    2: ("GPT 5.2", 0.5),
    3: ("GPT 5.2", 1),
    4: ("CS 4.6", 0),
    5: ("CS 4.6", 0.5),
    6: ("CS 4.6", 1),
}
mapped = df["Run"].map(run_map)
df["Model"] = mapped.str[0]
df["Temp"] = mapped.str[1]

def fmt_temp(t):
    if pd.isna(t):
        return None
    return str(int(t)) if float(t).is_integer() else str(t)

# 3) X-Achsen-Label (separat!)
df["x_label"] = df["Model"] + "\n t = " + df["Temp"].map(fmt_temp)

order = [
    "GPT 5.2\n t = 0",
    "GPT 5.2\n t = 0.5",
    "GPT 5.2\n t = 1",
    "CS 4.6\n t = 0",
    "CS 4.6\n t = 0.5",
    "CS 4.6\n t = 1",
]

# 4) max(p) Labels setzen (separat!)
max_p_label = [
    "6", "4", "3",
    "9", "9", "9", "9",
    "6",
    "9",
    "6",
    "3",
    "6", "6", "6", "6",
    "3",
    "6",
    "9",
    "9 (2)", "9", "9 (2)", "9", "9", "9 (2)",
    "3", "3", "3",
    "2", "2", "2"
]
df["max_p_label"] = max_p_label

# 5) Numerischen Wert für Farbe erzeugen
# Wenn du KEINE numerische Spalte hast, ziehe die Zahl aus max_p_label:
df["max_p_num"] = pd.to_numeric(df["max_p_label"].str.extract(r"(\d+)")[0], errors="coerce")

# 6) Matrizen bauen
value_matrix = (
    df.pivot(index="Experiment", columns="x_label", values="max_p_num")
      .reindex(columns=order)
      .sort_index()
)
label_matrix = (
    df.pivot(index="Experiment", columns="x_label", values="max_p_label")
      .reindex(columns=order)
      .sort_index()
)

# 7) Plot
sns.set_theme(style="white", context="paper", font_scale=1.1)
plt.figure(figsize=(7, 4))

sns.heatmap(
    value_matrix,
    annot=label_matrix,   # <-- Text-Labels (9(2))
    fmt="",
    cmap="RdYlGn_r",
    linewidths=0.5,
    cbar_kws={"label": "max penalty p, 9 (2) = 2 x penalty 9 "},
)

plt.title("Maximum penalty p for all experiments")
plt.xlabel("Model and Temperature")
plt.ylabel("Experiment")
plt.tight_layout()
plt.savefig("maxP.png", dpi=600, bbox_inches="tight")
plt.savefig("maxP.pdf", bbox_inches="tight")
plt.show()

Auswertung gesamt

In [ ]:
df = results

In [ ]:
mean_run = (
    df.groupby(["Experiment", "Run"])
      .agg(
          mean_E=("E", "mean"),
          mean_Ec=("Ec", "mean"),
          max_penalty=("max(p)", "mean")
      )
      .reset_index()
)

print(mean_run)

In [ ]:
mean_experiment = (
    df.groupby("Experiment")
      .agg(
          mean_E=("E", "mean"),
          mean_Ec=("Ec", "mean"),
          std_E=("E", "std"),
          runs=("Run", "count")
      )
      .reset_index()
)

print(mean_experiment)

In [ ]:
pivot_E = df.pivot_table(
    values="E",
    index="Experiment",
    columns="Run",
    aggfunc="mean"
)

print(pivot_E)

In [ ]:
df = pd.read_csv('results.csv', index_col=0, sep=',')

In [ ]:
run_map = {
    1: ("GPT 5.2", 0.0),
    2: ("GPT 5.2", 0.5),
    3: ("GPT 5.2", 1.0),
    4: ("Claude Sonnet 4.6", 0.0),
    5: ("Claude Sonnet 4.6", 0.5),
    6: ("Claude Sonnet 4.6", 1.0),
}

df["Model"] = df["Run"].map(lambda x: run_map[x][0])
df["Temp"] = df["Run"].map(lambda x: run_map[x][1])

model_perf = (
    df.groupby(["Model", "Temp"])
      .agg(
          mean_E=("E", "mean"),
          std_E=("E", "std"),
          mean_Ec=("Ec", "mean"),
          runs=("Run", "count")
      )
      .reset_index()
      .sort_values("mean_E")
)

print(model_perf)

In [ ]:
df